# Notebook 011: Custom PySpark UDFs vs Vectorized Pandas UDFs (PyArrow)

**Module:** Module 01: Core PySpark Foundations & DataFrame API  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 12: Debugging Code in Databricks & Video 25: Genie Code & Optimization  

---

## 1. Business Scenario & Real-World Context
A credit scoring model applies complex non-linear mathematical risk adjustments to customer accounts. The current implementation uses standard Python UDFs and causes severe CPU throttling and worker memory spikes.

## 2. Technical Requirement & Problem Statement
Implement the calculation using: (1) Standard Python UDF (`@udf`), (2) Native Spark SQL built-in expressions, and (3) Vectorized Pandas UDF (`@pandas_udf`). Compare execution time, serialization behavior, and memory footprint.

## 3. Spark & Databricks Architecture Notes
Standard Python UDFs serialize JVM row data to a Python worker process row-by-row via Py4J, incurring massive serialization and deserialization overhead and disabling Catalyst optimizations. Vectorized Pandas UDFs use Apache Arrow to transfer columnar batches into Pandas Series, achieving near-native C-speed vectorization.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_011_Custom_PySpark_UDFs_vs_Vectorized_Pandas_UDFs_(PyArrow)")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Creating Benchmark Financial Dataset
Prepare 50,000 accounts for algorithmic risk scoring.

In [ ]:
df_accounts = spark.range(1, 50001, 1, 4).select(
    F.col("id").alias("account_id"),
    (F.rand(seed=42) * 800 + 300).alias("credit_score"),
    (F.rand(seed=101) * 50000 + 1000).alias("debt_amount"),
    (F.rand(seed=202) * 100000 + 20000).alias("annual_income")
)

df_accounts.show(5)

## 6. Method 1: Standard Python UDF (Row-by-Row Serialization)
Define a standard Python function and wrap it with `@F.udf`.

In [ ]:
@F.udf(returnType=T.DoubleType())
def calculate_risk_standard(credit_score, debt, income):
    if income == 0 or credit_score is None:
        return 0.0
    dti = debt / income
    score_factor = (850 - credit_score) / 550.0
    return round((dti * 0.6) + (score_factor * 0.4), 4)

import time
t0 = time.time()
df_res1 = df_accounts.withColumn("risk_std", calculate_risk_standard("credit_score", "debt_amount", "annual_income"))
c1 = df_res1.filter(F.col("risk_std") > 0.5).count()
t1 = time.time() - t0
print(f"Standard UDF Count: {c1} in {t1:.3f} seconds")

## 7. Method 2: Vectorized Pandas UDF via PyArrow
Define a vectorized Arrow function operating over Pandas Series batches.

In [ ]:
import pandas as pd

@F.pandas_udf(T.DoubleType())
def calculate_risk_vectorized(credit_score: pd.Series, debt: pd.Series, income: pd.Series) -> pd.Series:
    dti = debt / income.replace(0, 1.0)
    score_factor = (850.0 - credit_score) / 550.0
    return ((dti * 0.6) + (score_factor * 0.4)).round(4)

t0 = time.time()
df_res2 = df_accounts.withColumn("risk_vec", calculate_risk_vectorized("credit_score", "debt_amount", "annual_income"))
c2 = df_res2.filter(F.col("risk_vec") > 0.5).count()
t2 = time.time() - t0
print(f"Vectorized Pandas UDF Count: {c2} in {t2:.3f} seconds (Speedup factor: {t1/max(t2, 0.001):.1f}x)")

## 9. Key Architectural Takeaways
- **Always favor native Spark SQL functions first before writing custom functions.**
- **If custom Python logic is unavoidable, always use `@pandas_udf` instead of standard `@udf`.**
- **Standard UDFs serialize data row-by-row through a Python subprocess, while Pandas UDFs batch data via shared Arrow memory.**

## 10. Senior Data Engineering Interview Questions
1. Why are standard Python UDFs considered a major performance bottleneck in PySpark?
2. How does Apache Arrow enable zero-copy data transfer in Pandas UDFs?
3. Can Catalyst Optimizer optimize or push down predicates past a Python UDF?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
